# OmegaPDF

Download manhwa panels from [OmegaScans](https://omegascans.org) and generate PDFs using the [OmegaAPI](https://omegaapi.vercel.app).

**Features:**
- Paste a URL to instantly download a chapter as PDF
- Concurrent panel downloads for speed
- Auto-retry on failed downloads
- Quality presets (Low / Medium / High DPI)
- Custom page ranges (e.g. pages 5-15 only)
- Merge multiple chapters into one PDF
- Batch download as separate PDFs
- Save directly to Google Drive
- **Send PDFs directly to Telegram via Bot**
- Progress bars with tqdm
- Thumbnail preview before downloading
- Browse trending series
- Search for any series

---

### Telegram Bot Setup
1. Create a bot via [@BotFather](https://t.me/BotFather) and copy the **bot token**
2. Get your **chat ID** — send a message to [@userinfobot](https://t.me/userinfobot)
3. **Start a chat with your bot** (send `/start`) so it can message you
4. Enter both values in the Setup cell below

---

In [ ]:
#@title Setup — Install dependencies & configure Telegram { display-mode: "form" }
!pip install -q requests Pillow tqdm

import requests
import io
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display, HTML

#@markdown ---
#@markdown ### Telegram Bot Configuration
TG_BOT_TOKEN = ""  #@param {type:"string"}
#@markdown _Bot token from @BotFather (e.g. `123456:ABC-DEF...`)_
TG_CHAT_ID = ""  #@param {type:"string"}
#@markdown _Your chat ID from @userinfobot (e.g. `987654321`)_

BASE_URL = "https://omegaapi.vercel.app"
API = f"{BASE_URL}/api/v1"
MAX_WORKERS = 8
MAX_RETRIES = 3
RETRY_BACKOFF = 1.0
TG_MAX_FILE_MB = 50  # Telegram bot upload limit

QUALITY_PRESETS = {
    "low": {"dpi": 72, "label": "Low (72 DPI)", "desc": "Fast, small files"},
    "medium": {"dpi": 150, "label": "Medium (150 DPI)", "desc": "Balanced"},
    "high": {"dpi": 300, "label": "High (300 DPI)", "desc": "Print quality, large files"},
}

def api_get(path, params=None):
    r = requests.get(f"{API}{path}", params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def download_image_retry(url):
    for attempt in range(MAX_RETRIES):
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            return r.content
        except (requests.HTTPError, requests.ConnectionError, requests.Timeout):
            if attempt == MAX_RETRIES - 1:
                raise
            time.sleep(RETRY_BACKOFF * (2 ** attempt))

def download_images_concurrent(urls):
    results = {}
    pbar = tqdm(total=len(urls), desc="Downloading panels", unit="panel")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(download_image_retry, url): i for i, url in enumerate(urls)}
        for future in as_completed(futures):
            idx = futures[future]
            results[idx] = future.result()
            pbar.update(1)
    pbar.close()
    return [results[i] for i in range(len(urls))]

def images_to_pdf(image_bytes_list, output_path, title=None, author=None, subject=None, dpi=150):
    pages = []
    for b in image_bytes_list:
        img = Image.open(io.BytesIO(b))
        if img.mode == "RGBA":
            img = img.convert("RGB")
        pages.append(img)

    pdf_info = {}
    if title: pdf_info["Title"] = title
    if author: pdf_info["Author"] = author
    if subject: pdf_info["Subject"] = subject

    first = pages[0]
    rest = pages[1:] if len(pages) > 1 else []
    first.save(
        output_path, format="PDF", save_all=True, append_images=rest,
        resolution=dpi, quality=90, pdf_info=pdf_info or None,
    )
    for img in pages:
        img.close()
    return output_path

def fetch_chapter(slug, chapter_slug):
    data = api_get(f"/chapter/{slug}/{chapter_slug}")
    if not data.get("success"):
        raise RuntimeError(data.get("error", "Chapter not found"))
    info = data["data"]
    urls = info.get("images", [])
    if not urls:
        raise RuntimeError("No images found")
    series_title = info.get("series", {}).get("title", slug)
    ch_name = info.get("name", chapter_slug)
    return series_title, ch_name, urls

def safe_filename(text):
    return re.sub(r'[^\w\s-]', '', text).strip().replace(' ', '_')

def send_to_telegram(file_path, caption=None, bot_token=None, chat_id=None):
    """Send a file to Telegram via Bot API. Returns True on success."""
    token = bot_token or TG_BOT_TOKEN
    cid = chat_id or TG_CHAT_ID
    if not token or not cid:
        print("Error: TG_BOT_TOKEN and TG_CHAT_ID not configured. Set them in the Setup cell.")
        return False

    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    if size_mb > TG_MAX_FILE_MB:
        print(f"File too large ({size_mb:.1f} MB). Telegram limit is {TG_MAX_FILE_MB} MB.")
        return False

    url = f"https://api.telegram.org/bot{token}/sendDocument"
    fname = os.path.basename(file_path)
    payload = {"chat_id": cid}
    if caption:
        payload["caption"] = caption[:1024]  # Telegram caption limit

    print(f"Sending to Telegram ({size_mb:.1f} MB)...")
    with open(file_path, "rb") as f:
        resp = requests.post(
            url,
            data=payload,
            files={"document": (fname, f, "application/pdf")},
            timeout=120,
        )

    if resp.status_code == 200 and resp.json().get("ok"):
        print(f"Sent to Telegram successfully!")
        return True
    else:
        err = resp.json().get("description", resp.text) if resp.headers.get("content-type", "").startswith("application/json") else resp.text
        print(f"Telegram error: {err}")
        return False

def send_bytes_to_telegram(file_bytes, filename, caption=None, bot_token=None, chat_id=None):
    """Send raw bytes as a file to Telegram. Returns True on success."""
    token = bot_token or TG_BOT_TOKEN
    cid = chat_id or TG_CHAT_ID
    if not token or not cid:
        print("Error: TG_BOT_TOKEN and TG_CHAT_ID not configured.")
        return False

    size_mb = len(file_bytes) / (1024 * 1024)
    if size_mb > TG_MAX_FILE_MB:
        print(f"File too large ({size_mb:.1f} MB). Telegram limit is {TG_MAX_FILE_MB} MB.")
        return False

    url_api = f"https://api.telegram.org/bot{token}/sendDocument"
    payload = {"chat_id": cid}
    if caption:
        payload["caption"] = caption[:1024]

    print(f"Sending to Telegram ({size_mb:.1f} MB)...")
    resp = requests.post(
        url_api,
        data=payload,
        files={"document": (filename, file_bytes, "application/pdf")},
        timeout=120,
    )

    if resp.status_code == 200 and resp.json().get("ok"):
        print(f"Sent to Telegram successfully!")
        return True
    else:
        err = resp.json().get("description", resp.text) if resp.headers.get("content-type", "").startswith("application/json") else resp.text
        print(f"Telegram error: {err}")
        return False

def test_telegram():
    """Test Telegram bot connection by sending a test message."""
    if not TG_BOT_TOKEN or not TG_CHAT_ID:
        print("TG_BOT_TOKEN or TG_CHAT_ID not set.")
        return False
    url = f"https://api.telegram.org/bot{TG_BOT_TOKEN}/sendMessage"
    resp = requests.post(url, data={
        "chat_id": TG_CHAT_ID,
        "text": "OmegaPDF connected! Ready to send manga PDFs.",
    }, timeout=15)
    if resp.status_code == 200 and resp.json().get("ok"):
        print("Telegram bot works! Check your chat for a test message.")
        return True
    else:
        err = resp.json().get("description", "Unknown error")
        print(f"Telegram error: {err}")
        return False

tg_ok = bool(TG_BOT_TOKEN and TG_CHAT_ID)
if tg_ok:
    print(f"Ready! Telegram bot configured (token: ...{TG_BOT_TOKEN[-6:]})")
else:
    print("Ready! Telegram not configured — PDFs will download locally.")
    print("To enable Telegram: fill in TG_BOT_TOKEN and TG_CHAT_ID above.")

In [ ]:
#@title Test Telegram bot connection { display-mode: "form" }
#@markdown _Run this after filling in your bot token and chat ID to verify it works._

test_telegram()

In [ ]:
#@title Download from URL — just paste and run! { display-mode: "form" }
url = "https://omegascans.org/series/manitto/chapter-82"  #@param {type:"string"}
quality = "medium"  #@param ["low", "medium", "high"]
page_range = ""  #@param {type:"string"}
#@markdown _Page range format: `5-15` (pages 5 through 15). Leave blank for all._
send_to_tg = False  #@param {type:"boolean"}
#@markdown _Send PDF directly to Telegram instead of downloading locally._
save_to_drive = False  #@param {type:"boolean"}
#@markdown _Save PDF to Google Drive (requires mounting)._

match = re.search(r"omegascans\.org/series/([^/]+)/chapter-(\d+)", url)
if not match:
    print("Invalid URL. Expected: https://omegascans.org/series/{slug}/chapter-{number}")
else:
    slug = match.group(1)
    chapter_slug = f"chapter-{match.group(2)}"
    series_title, ch_name, image_urls = fetch_chapter(slug, chapter_slug)

    if page_range.strip():
        parts = page_range.split("-")
        start, end = int(parts[0]), int(parts[1])
        image_urls = image_urls[max(0, start-1):end]
        print(f"Using pages {start}-{end}")

    print(f"Series: {series_title}")
    print(f"Chapter: {ch_name}")
    print(f"Pages: {len(image_urls)}")

    # Show thumbnail preview
    print("\nPreview (first page):")
    thumb = requests.get(image_urls[0], timeout=30).content
    thumb_img = Image.open(io.BytesIO(thumb))
    display(thumb_img.copy())
    thumb_img.close()

    dpi = QUALITY_PRESETS[quality]["dpi"]
    image_bytes = download_images_concurrent(image_urls)

    fname = f"{safe_filename(series_title)}_{chapter_slug}.pdf"
    images_to_pdf(
        image_bytes, fname,
        title=f"{series_title} — {ch_name}",
        author="OmegaPDF", subject=series_title, dpi=dpi,
    )
    size_mb = os.path.getsize(fname) / (1024 * 1024)
    print(f"\nPDF ready: {fname} ({size_mb:.2f} MB, {len(image_urls)} pages)")

    if send_to_tg:
        send_to_telegram(fname, caption=f"{series_title} — {ch_name}")
    elif save_to_drive:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_path = f"/content/drive/MyDrive/{fname}"
        import shutil
        shutil.copy2(fname, drive_path)
        print(f"Saved to Drive: {drive_path}")
    else:
        from google.colab import files
        files.download(fname)

In [ ]:
#@title Search for a series { display-mode: "form" }
query = "solo leveling"  #@param {type:"string"}

results = api_get("/search", params={"q": query})
if results.get("success") and results["data"]:
    for i, s in enumerate(results["data"][:10]):
        chapters = s.get("chaptersCount", "?")
        status = s.get("status", "")
        print(f"{i+1}. {s['title']}  [{status}] — {chapters} chapters  —  slug: {s['slug']}")
else:
    print("No results found. Try a different search term.")

In [ ]:
#@title Browse trending & popular series { display-mode: "form" }
page = 1  #@param {type:"integer"}
per_page = 15  #@param {type:"slider", min:5, max:50, step:5}

data = api_get("/series", params={"page": page, "perPage": per_page})
if data.get("success") and data["data"]:
    print(f"Page {page} — {len(data['data'])} series\n")
    for i, s in enumerate(data["data"], 1):
        views = s.get("totalViews", 0)
        views_str = f"{views/1_000_000:.1f}M" if views >= 1_000_000 else f"{views:,}"
        badge = f" [{s['badge']}]" if s.get('badge') else ""
        rating = s.get('rating', 0)
        chapters = s.get('chaptersCount', '?')
        status = s.get('status', '')
        print(f"{i:2d}. {s['title']}{badge}")
        print(f"    Rating: {rating} | Views: {views_str} | {chapters} ch | {status}")
        print(f"    slug: {s['slug']}")
        print()
else:
    print("No series found.")

In [ ]:
#@title List chapters for a series { display-mode: "form" }
series_slug = "solo-leveling"  #@param {type:"string"}

series = api_get(f"/series/{series_slug}")
if series.get("success"):
    data = series["data"]
    print(f"Title: {data['title']}")
    print(f"Status: {data.get('status', 'N/A')}")
    print(f"Chapters: {data.get('chaptersCount', len(data.get('chapters', [])))}")
    print(f"\nAvailable chapters:")
    chapters = data.get("chapters", [])
    for ch in chapters[:30]:
        free = "[Free]" if ch.get("isFree") else "[Paid]"
        print(f"  {ch['name']} {free}  —  slug: {ch['slug']}")
    if len(chapters) > 30:
        print(f"  ... and {len(chapters) - 30} more chapters")
else:
    print(f"Error: {series.get('error', 'Series not found')}")

In [ ]:
#@title Download chapter by slug { display-mode: "form" }
slug = "solo-leveling"  #@param {type:"string"}
chapter = "chapter-1"  #@param {type:"string"}
output_name = "Solo_Leveling_Ch1"  #@param {type:"string"}
quality = "medium"  #@param ["low", "medium", "high"]
page_range = ""  #@param {type:"string"}
send_to_tg = False  #@param {type:"boolean"}
#@markdown _Send PDF directly to Telegram instead of downloading locally._
save_to_drive = False  #@param {type:"boolean"}

series_title, ch_name, image_urls = fetch_chapter(slug, chapter)

if page_range.strip():
    parts = page_range.split("-")
    start, end = int(parts[0]), int(parts[1])
    image_urls = image_urls[max(0, start-1):end]
    print(f"Using pages {start}-{end}")

print(f"Series: {series_title}")
print(f"Chapter: {ch_name}")
print(f"Pages: {len(image_urls)}")

# Thumbnail preview
print("\nPreview (first page):")
thumb = requests.get(image_urls[0], timeout=30).content
thumb_img = Image.open(io.BytesIO(thumb))
display(thumb_img.copy())
thumb_img.close()

dpi = QUALITY_PRESETS[quality]["dpi"]
image_bytes = download_images_concurrent(image_urls)

fname = f"{output_name}.pdf"
images_to_pdf(
    image_bytes, fname,
    title=f"{series_title} — {ch_name}",
    author="OmegaPDF", subject=series_title, dpi=dpi,
)
size_mb = os.path.getsize(fname) / (1024 * 1024)
print(f"\nPDF ready: {fname} ({size_mb:.2f} MB, {len(image_urls)} pages)")

if send_to_tg:
    send_to_telegram(fname, caption=f"{series_title} — {ch_name}")
elif save_to_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_path = f"/content/drive/MyDrive/{fname}"
    import shutil
    shutil.copy2(fname, drive_path)
    print(f"Saved to Drive: {drive_path}")
else:
    from google.colab import files
    files.download(fname)

In [ ]:
#@title Merge multiple chapters into one PDF { display-mode: "form" }
merge_slug = "solo-leveling"  #@param {type:"string"}
start_ch = 1  #@param {type:"integer"}
end_ch = 3  #@param {type:"integer"}
merge_quality = "medium"  #@param ["low", "medium", "high"]
send_to_tg = False  #@param {type:"boolean"}
#@markdown _Send PDF directly to Telegram instead of downloading locally._
save_to_drive = False  #@param {type:"boolean"}

dpi = QUALITY_PRESETS[merge_quality]["dpi"]
all_bytes = []
series_title = merge_slug

for ch_num in range(start_ch, end_ch + 1):
    ch_id = f"chapter-{ch_num}"
    print(f"Fetching {ch_id}...")
    try:
        series_title, ch_name, urls = fetch_chapter(merge_slug, ch_id)
        ch_bytes = download_images_concurrent(urls)
        all_bytes.extend(ch_bytes)
        print(f"  {ch_name}: {len(urls)} pages")
    except Exception as e:
        print(f"  Skipping {ch_id}: {e}")

if all_bytes:
    safe = safe_filename(series_title)
    fname = f"{safe}_ch{start_ch}-{end_ch}_merged.pdf"
    images_to_pdf(
        all_bytes, fname,
        title=f"{series_title} — Chapters {start_ch}-{end_ch}",
        author="OmegaPDF", subject=series_title, dpi=dpi,
    )
    size_mb = os.path.getsize(fname) / (1024 * 1024)
    print(f"\nMerged PDF: {fname} ({size_mb:.2f} MB, {len(all_bytes)} pages)")

    if send_to_tg:
        send_to_telegram(fname, caption=f"{series_title} — Ch {start_ch}-{end_ch}")
    elif save_to_drive:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_path = f"/content/drive/MyDrive/{fname}"
        import shutil
        shutil.copy2(fname, drive_path)
        print(f"Saved to Drive: {drive_path}")
    else:
        from google.colab import files
        files.download(fname)
else:
    print("No chapters downloaded.")

In [ ]:
#@title Batch download — send to Telegram { display-mode: "form" }
batch_slug = "solo-leveling"  #@param {type:"string"}
start_ch = 1  #@param {type:"integer"}
end_ch = 5  #@param {type:"integer"}
batch_quality = "medium"  #@param ["low", "medium", "high"]
send_to_tg = True  #@param {type:"boolean"}
#@markdown _Send each chapter PDF directly to Telegram._
zip_download = False  #@param {type:"boolean"}
#@markdown _Or zip all PDFs into a single local download._

dpi = QUALITY_PRESETS[batch_quality]["dpi"]
pdf_files = []

for ch_num in range(start_ch, end_ch + 1):
    ch_id = f"chapter-{ch_num}"
    print(f"\n{'='*40}")
    print(f"Processing {ch_id}...")
    try:
        series_title, ch_name, urls = fetch_chapter(batch_slug, ch_id)
        image_bytes = download_images_concurrent(urls)
        fname = f"{batch_slug}_{ch_id}.pdf"
        images_to_pdf(
            image_bytes, fname,
            title=f"{series_title} — {ch_name}",
            author="OmegaPDF", subject=series_title, dpi=dpi,
        )
        size_mb = os.path.getsize(fname) / (1024 * 1024)
        print(f"  Saved: {fname} ({size_mb:.2f} MB, {len(urls)} pages)")
        pdf_files.append(fname)

        if send_to_tg:
            send_to_telegram(fname, caption=f"{series_title} — {ch_name}")
            time.sleep(1)  # rate limit between sends
    except Exception as e:
        print(f"  Error: {e}")

print(f"\nDone! {len(pdf_files)} PDFs processed.")

if pdf_files and not send_to_tg:
    if zip_download:
        import zipfile
        zip_name = f"{batch_slug}_ch{start_ch}-{end_ch}.zip"
        with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
            for f in pdf_files:
                zf.write(f)
        zip_mb = os.path.getsize(zip_name) / (1024 * 1024)
        print(f"Zip: {zip_name} ({zip_mb:.2f} MB)")
        from google.colab import files
        files.download(zip_name)
    else:
        from google.colab import files
        for f in pdf_files:
            files.download(f)

In [ ]:
#@title Send any local PDF to Telegram { display-mode: "form" }
#@markdown _Use this to send a PDF that's already on disk (e.g. from a previous download)._
file_path = "Solo_Leveling_chapter-1.pdf"  #@param {type:"string"}
caption = ""  #@param {type:"string"}
#@markdown _Optional caption for the Telegram message._

if not os.path.exists(file_path):
    print(f"File not found: {file_path}")
else:
    send_to_telegram(file_path, caption=caption or None)